In [1]:
from collections import deque
import heapq

# Campus graph with walking distances in meters
graph = {
    'A': {'B': 100, 'C': 150},
    'B': {'A': 100, 'D': 120, 'E': 180},
    'C': {'A': 150, 'E': 100, 'F': 200},
    'D': {'B': 120, 'G': 150},
    'E': {'B': 180, 'C': 100, 'F': 100, 'G': 160},
    'F': {'C': 200, 'E': 100, 'H': 120},
    'G': {'D': 150, 'E': 160, 'H': 140},
    'H': {'F': 120, 'G': 140}
}

start = 'A'
goal = 'H'

# Heuristic estimates to the goal (meters)
heuristic = {
    'A': 400,
    'B': 350,
    'C': 300,
    'D': 250,
    'E': 200,
    'F': 100,
    'G': 120,
    'H': 0
}


def path_cost(path):
    total = 0
    for i in range(len(path) - 1):
        total += graph[path[i]][path[i + 1]]
    return total


# 1. Breadth-First Search
def bfs(start, goal):
    queue = deque([[start]])
    visited = {start}
    order = []

    while queue:
        path = queue.popleft()
        node = path[-1]
        order.append(node)

        if node == goal:
            return order, path, path_cost(path)

        for neighbor in sorted(graph[node]):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])

    return order, [], 0


# 2. Depth-First Search
def dfs(start, goal):
    visited = set()
    order = []

    def search(node, path):
        visited.add(node)
        order.append(node)

        if node == goal:
            return path

        for neighbor in sorted(graph[node]):
            if neighbor not in visited:
                result = search(neighbor, path + [neighbor])
                if result:
                    return result

        return None

    result = search(start, [start])

    if result:
        return order, result, path_cost(result)

    return order, [], 0


# 3. Greedy Best-First Search
def greedy(start, goal):
    counter = 0
    pq = [(heuristic[start], counter, start, [start])]
    visited = set()
    order = []

    while pq:
        h, _, node, path = heapq.heappop(pq)

        if node in visited:
            continue

        visited.add(node)
        order.append(node)

        if node == goal:
            return order, path, path_cost(path)

        for neighbor in sorted(graph[node]):
            if neighbor not in visited:
                counter += 1
                heapq.heappush(
                    pq,
                    (heuristic[neighbor], counter,
                     neighbor, path + [neighbor])
                )

    return order, [], 0


# 4. A* Search
def astar(start, goal):
    counter = 0
    pq = [(heuristic[start], counter, start, 0, [start])]
    best_g = {start: 0}
    order = []

    while pq:
        f, _, node, g, path = heapq.heappop(pq)

        if g != best_g[node]:
            continue

        order.append(node)

        if node == goal:
            return order, path, g

        for neighbor in sorted(graph[node]):
            new_g = g + graph[node][neighbor]

            if new_g < best_g.get(neighbor, float('inf')):
                best_g[neighbor] = new_g
                new_f = new_g + heuristic[neighbor]
                counter += 1

                heapq.heappush(
                    pq,
                    (new_f, counter, neighbor,
                     new_g, path + [neighbor])
                )

    return order, [], 0


# Display results
algorithms = [
    ("BFS", bfs),
    ("DFS", dfs),
    ("Greedy Best-First Search", greedy),
    ("A* Search", astar)
]

for name, algorithm in algorithms:
    order, path, cost = algorithm(start, goal)

    print("\nAlgorithm:", name)
    print("Initial State:", start)
    print("Goal State:", goal)
    print("Node Expansion Order:", order)
    print("Final Path:", " -> ".join(path))
    print("Total Path Cost:", cost, "meters")
    print("Number of Nodes Expanded:", len(order))


Algorithm: BFS
Initial State: A
Goal State: H
Node Expansion Order: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Final Path: A -> C -> F -> H
Total Path Cost: 470 meters
Number of Nodes Expanded: 8

Algorithm: DFS
Initial State: A
Goal State: H
Node Expansion Order: ['A', 'B', 'D', 'G', 'E', 'C', 'F', 'H']
Final Path: A -> B -> D -> G -> E -> C -> F -> H
Total Path Cost: 950 meters
Number of Nodes Expanded: 8

Algorithm: Greedy Best-First Search
Initial State: A
Goal State: H
Node Expansion Order: ['A', 'C', 'F', 'H']
Final Path: A -> C -> F -> H
Total Path Cost: 470 meters
Number of Nodes Expanded: 4

Algorithm: A* Search
Initial State: A
Goal State: H
Node Expansion Order: ['A', 'B', 'C', 'E', 'F', 'D', 'H']
Final Path: A -> C -> F -> H
Total Path Cost: 470 meters
Number of Nodes Expanded: 7
